# はじめに

このノートでは、下記のKaggleノートの内容を参考にグラフニューラルネットワークへの理解を深める。

- [Kaggle: Anti Money Laundering Detection with GNN](https://www.kaggle.com/code/issacchanjj/anti-money-laundering-detection-with-gnn)
- [HI-Small_Trans.csv · bbfizp/AMLSim-HI-Small at main](https://huggingface.co/datasets/bbfizp/AMLSim-HI-Small/blob/main/HI-Small_Trans.csv)

扱っているのはマネーロンダリングのノード予測であり、取引データをグラフに変換して、各ノードである口座がマネーロンダリングに関与しているかどうかをノード分類で判定する、という流れになっている。

## データ

マネーロンダリングということもあり、極端に不均衡な状態となっている。そのため、ここでのノートはあくまでもマネーロンダリングのようなデータをグラフニューラルネットワークで学習するためのデモくらいに考えるのがよいかも。

In [1]:
import datetime
import os
from typing import Callable, Optional
import pandas as pd
from sklearn import preprocessing
import numpy as np
import torch

from torch_geometric.data import (
    Data,
    InMemoryDataset
)

pd.set_option('display.max_columns', None)
path = './HI-Small_Trans.csv'
df = pd.read_csv(path)

/Users/aki/miniforge3/envs/pyg311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


およそ500万行、11列で構成されており、各口座の送金記録が保存されている。カラムの説明は下記の通り。

- 送金元銀行 `From Bank`
- 送金先銀行 `To Bank`
- 送金元口座 `Account`
- 送金先口座 `Account.1`
- 支払金額 `Amount Paid`
- 受取金額 `Amount Received`
- 支払通貨 `Payment Currency`
- 受取通貨 `Receiving Currency`
- 決済形式 `Payment Format`
- 時刻 `Timestamp`
- マネーロンダリングかどうか `Is Laundering`

In [2]:
df

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.340000,US Dollar,3697.340000,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.010000,US Dollar,0.010000,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.570000,US Dollar,14675.570000,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.970000,US Dollar,2806.970000,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.970000,US Dollar,36682.970000,US Dollar,Reinvestment,0
...,...,...,...,...,...,...,...,...,...,...,...
5078340,2022/09/10 23:57,54219,8148A6631,256398,8148A8711,0.154978,Bitcoin,0.154978,Bitcoin,Bitcoin,0
5078341,2022/09/10 23:35,15,8148A8671,256398,8148A8711,0.108128,Bitcoin,0.108128,Bitcoin,Bitcoin,0
5078342,2022/09/10 23:52,154365,8148A6771,256398,8148A8711,0.004988,Bitcoin,0.004988,Bitcoin,Bitcoin,0
5078343,2022/09/10 23:46,256398,8148A6311,256398,8148A8711,0.038417,Bitcoin,0.038417,Bitcoin,Bitcoin,0


nullは存在していない。

In [3]:
print(df.isnull().sum())

Timestamp             0
From Bank             0
Account               0
To Bank               0
Account.1             0
Amount Received       0
Receiving Currency    0
Amount Paid           0
Payment Currency      0
Payment Format        0
Is Laundering         0
dtype: int64


各取引の支払額`Amount Paid`と受取額`Amount Received`を表す2つの列があり、同じ値が共有されていそうに見えるが、取引手数料や異なる通貨間の取引の影響によって一致していないレコードが存在する。

In [4]:
print('Amount Received equals to Amount Paid:')
print(df['Amount Received'].equals(df['Amount Paid']))
print('Receiving Currency equals to Payment Currency:')
print(df['Receiving Currency'].equals(df['Payment Currency']))

Amount Received equals to Amount Paid:
False
Receiving Currency equals to Payment Currency:
False


異なる通貨間の取引が関係しており、`Receiving Currency`、`Payment Currency`を見るとわかる。

In [5]:
df.loc[~(df['Amount Received'] == df['Amount Paid'])]

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
1173,2022/09/01 00:22,1362,80030A870,1362,80030A870,52.110000,Euro,61.06,US Dollar,ACH,0
7156,2022/09/01 00:28,11318,800C51010,11318,800C51010,76.060000,Euro,89.12,US Dollar,ACH,0
7925,2022/09/01 00:12,795,800D98770,795,800D98770,17.690000,Australian Dollar,12.52,US Dollar,ACH,0
8467,2022/09/01 00:01,1047,800E92CF0,1047,800E92CF0,19.430000,Euro,22.77,US Dollar,ACH,0
11529,2022/09/01 00:22,11157,80135FFC0,11157,80135FFC0,98.340000,Euro,115.24,US Dollar,ACH,0
...,...,...,...,...,...,...,...,...,...,...,...
5078167,2022/09/10 23:30,23537,803949A90,23537,803949A90,26421.500000,Shekel,7823.96,US Dollar,ACH,0
5078234,2022/09/10 23:59,16163,803638A90,16163,803638A90,47517.490000,Saudi Riyal,12667.62,US Dollar,ACH,0
5078236,2022/09/10 23:55,16163,803638A90,16163,803638A90,11329.850000,Saudi Riyal,3020.41,US Dollar,ACH,0
5078316,2022/09/10 23:44,215064,808F06E11,215064,808F06E10,0.000006,Bitcoin,0.07,US Dollar,ACH,0


## データ前処理

#データ前処理では、以下の変換を行う。

1. `Timestamp` を min-max 正規化
2. 口座番号に銀行コードを追加して、各口座に一意の ID を作成（`From Bank` + `Account`, `To Bank` + `Account.1`）
3. 受取口座、受取金額、通貨の情報を含む `receiving_df` を作成（`Account.1`, `Amount Received`, `Receiving Currency`）
4. 支払口座、支払金額、通貨の情報を含む `paying_df` を作成（`Account`, `Amount Paid`, `Payment Currency`）
5. すべての取引で使用された通貨（`Receiving Currency`、`Payment Currency`）のリストを作成
6. sklearn の LabelEncoder を使用して、`Payment Format`, `Payment Currency`, `Receiving Currency` をクラスごとにラベル付け

最終的には、各口座の特徴量を作るために、「その口座が支払った履歴(`paying_df`)」「その口座が受け取った履歴( `receiving_df`)」を分けておく。こうすることで、簡単にある口座について、どの通貨で、平均どれくらい支払うか。平均どれくらい受け取るかを別々に集計できる。

In [6]:
# カテゴリ列を受け取り、行ごとにカテゴリの数値を割り当てる関数
def df_label_encoder(df, columns):
        le = preprocessing.LabelEncoder()
        for i in columns:
            df[i] = le.fit_transform(df[i].astype(str))
        return df

def preprocess(df):
        df = df_label_encoder(df,['Payment Format', 'Payment Currency', 'Receiving Currency'])
        df['Timestamp'] = pd.to_datetime(df['Timestamp'])
        df['Timestamp'] = df['Timestamp'].apply(lambda x: x.value)
        df['Timestamp'] = (df['Timestamp']-df['Timestamp'].min())/(df['Timestamp'].max()-df['Timestamp'].min())

        df['Account'] = df['From Bank'].astype(str) + '_' + df['Account']
        df['Account.1'] = df['To Bank'].astype(str) + '_' + df['Account.1']
        df = df.sort_values(by=['Account'])
        receiving_df = df[['Account.1', 'Amount Received', 'Receiving Currency']]
        paying_df = df[['Account', 'Amount Paid', 'Payment Currency']]
        receiving_df = receiving_df.rename({'Account.1': 'Account'}, axis=1)
        currency_ls = sorted(df['Receiving Currency'].unique())

        return df, receiving_df, paying_df, currency_ls

In [7]:
df, receiving_df, paying_df, currency_ls = preprocess(df = df)

In [8]:
print(f'df.shape: {df.shape}, df.columns: {df.columns}')
print('-'*100)
print(f'receiving_df.shape: {receiving_df.shape}, receiving_df.columns: {receiving_df.columns}')
print('-'*100)
print(f'paying_df.shape: {paying_df.shape}, paying_df.columns: {paying_df.columns}')
print('-'*100)
print(f'currency_ls: {len(currency_ls)} ')

df.shape: (5078345, 11), df.columns: Index(['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1',
       'Amount Received', 'Receiving Currency', 'Amount Paid',
       'Payment Currency', 'Payment Format', 'Is Laundering'],
      dtype='str')
----------------------------------------------------------------------------------------------------
receiving_df.shape: (5078345, 3), receiving_df.columns: Index(['Account', 'Amount Received', 'Receiving Currency'], dtype='str')
----------------------------------------------------------------------------------------------------
paying_df.shape: (5078345, 3), paying_df.columns: Index(['Account', 'Amount Paid', 'Payment Currency'], dtype='str')
----------------------------------------------------------------------------------------------------
currency_ls: 15 


In [9]:
print(receiving_df.head())
print('-'*100)
print(paying_df.head())
print('-'*100)
print(currency_ls)

                 Account  Amount Received  Receiving Currency
4278714  29467_803E020C0        787197.11                  13
2798190  29467_803E020C0        787197.11                  13
2798191  29467_803E020C0        681262.19                  13
3918769  29467_803E020C0        681262.19                  13
213094   10057_803A115E0        146954.27                  13
----------------------------------------------------------------------------------------------------
                 Account  Amount Paid  Payment Currency
4278714  10057_803A115E0    787197.11                13
2798190  10057_803A115E0    787197.11                13
2798191  10057_803A115E0    681262.19                13
3918769  10057_803A115E0    681262.19                13
213094   10057_803A115E0    146954.27                13
----------------------------------------------------------------------------------------------------
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]


## ノード特徴量

`Is Laundering`ラベルが取引単位で与えられていたが、ここでは取引単位から口座単位に情報を集約し、口座=ノード単位の判定問題に変換する。とはいえ、何をしているのかパッと見てわからないので、行ごとに説明する。

In [10]:
def get_all_account(df):
        ldf = df[['Account', 'From Bank']] # お金を出した側
        rdf = df[['Account.1', 'To Bank']] # お金を受け取った側
        # 不正取引に関わった口座は両方とも不正扱いにする
        suspicious = df[df['Is Laundering']==1] # 不正取引に関与している取引
        s1 = suspicious[['Account', 'Is Laundering']] # 不正取引に関与している取引のうち、お金を出した側
        s2 = suspicious[['Account.1', 'Is Laundering']] # 不正取引に関与している取引のうち、お金を受け取った側
        s2 = s2.rename({'Account.1': 'Account'}, axis=1)
        suspicious = pd.concat([s1, s2], join='outer')
        # 不正取引に関与している口座は重複しているので、重複を削除
        suspicious = suspicious.drop_duplicates()

        ldf = ldf.rename({'From Bank': 'Bank'}, axis=1)
        rdf = rdf.rename({'Account.1': 'Account', 'To Bank': 'Bank'}, axis=1)
        df = pd.concat([ldf, rdf], join='outer')
        df = df.drop_duplicates()

        df['Is Laundering'] = 0
        df.set_index('Account', inplace=True)
        df.update(suspicious.set_index('Account'))
        df = df.reset_index()
        return df

次のような3件の取引データを考える。

- 1行目：A → B（正常）
- 2行目：B → C（不正）
- 3行目：D → A（正常）

|行 | Account | From Bank | Account.1 | To Bank | Is Laundering|
|---|---------|-----------|-----------|---------|----------------|
|1  | A       | B1        | B         | B2      | 0               |
|2  | B       | B2        | C         | B3      | 1               |
|3  | D       | B1        | A         | B1      | 0               |

`ldf = df[['Account', 'From Bank']]`で`ldf`（送金側） を作成し、

|Account | From Bank→Bank
|--------|-----------
|A       | B1
|B       | B2
|D       | B1

`rdf = df[['Account.1', 'To Bank']]`で`rdf`（受取側）を作成する。矢印は後ほど行われる`rename`を意味する。

|Account.1 →Account | To Bank→Bank
|----------|---------
|B         | B2
|C         | B3
|A         | B1

`suspicious = df[df['Is Laundering']==1]`で不正取引を取り出す。

|Account | Account.1 | Is Laundering
|--------|-----------|----------------
|B       | C         | 1

`s1 = suspicious[['Account', 'Is Laundering']]`で`s1`（送金側）で不正に関わった口座を抽出する。

|Account | Is Laundering
|--------|---------------
|B       | 1

`s2 = suspicious[['Account.1', 'Is Laundering']]`で`s2`（受取側）で不正に関わった口座を抽出する。

|Account.1→Account | Is Laundering
|----------|---------------
|C         | 1

`suspicious = pd.concat([s1, s2], join='outer')`で送金側`s1`と受取側`s2`を結合して1つにまとめる。`suspicious = suspicious.drop_duplicates()`で重複を削除。


|Account | Is Laundering
|--------|---------------
|B       | 1
|C         | 1


そして、`df = pd.concat([ldf, rdf], join='outer'), df = df.drop_duplicates()`で送金側、受取側の口座をまとめる。ここで登場したすべての口座がリストアップされる。初期ラベルを全部0にしてから、疑いのある講座は`suspicious`データをもとに更新。

Account | Bank | Is Laundering
--------|------|----------------
A       | B1   | 0
B       | B2   | 0 → 1
C       | B3   | 0 → 1
D       | B1   | 0




「不正取引に1回でも関わった口座」= 不正口座として扱うため、取引ラベルを口座ラベルへ変換している。これは問題設定には注意が必要。なぜなら、ある口座が一度だけ不正取引に巻き込まれたからといって、その口座自体を「不正口座」とみなしてよいのか、という設計上の懸念があるが、ここでは問題ないものとする。

In [11]:
accounts = get_all_account(df)
accounts

,Account,Bank,Is Laundering
0,10057_803A115E0,10057,0
1,10057_803AA8E90,10057,0
2,10057_803AAB430,10057,0
3,10057_803AACE20,10057,0
4,10057_803AB4F70,10057,0
...,...,...,...
515083,16792_8061CF100,16792,0
515084,17554_8071B9990,17554,0
515085,217125_8065EA0B1,217125,0
515086,217824_806BEF7C1,217824,0


## ノードの特徴量

ノードの特徴量として、銀行コード、通貨ごとの平均支払額、通貨ごとの平均受取額を集計して利用する。口座Aについて、支払履歴が次のようだったとする。

- USDで100支払い
- USDで300支払い
- EURで200支払い

このとき、

- avg paid USD = (100+300)/2=200
- avg paid EUR = 200
- avg paid JPY = 0

のようになる。受け取りは逆の関係になる。取引履歴を、口座ごとの統計量に圧縮する。

In [12]:
def paid_currency_aggregate(currency_ls, paying_df, accounts):
        for i in currency_ls:
            temp = paying_df[paying_df['Payment Currency'] == i]
            accounts['avg paid '+str(i)] = temp['Amount Paid'].groupby(temp['Account']).transform('mean')
        return accounts

def received_currency_aggregate(currency_ls, receiving_df, accounts):
    for i in currency_ls:
        temp = receiving_df[receiving_df['Receiving Currency'] == i]
        accounts['avg received '+str(i)] = temp['Amount Received'].groupby(temp['Account']).transform('mean')
    accounts = accounts.fillna(0)
    return accounts

def get_node_attr(currency_ls, paying_df,receiving_df, accounts):
        node_df = paid_currency_aggregate(currency_ls, paying_df, accounts)
        node_df = received_currency_aggregate(currency_ls, receiving_df, node_df)
        node_label = torch.from_numpy(node_df['Is Laundering'].values).to(torch.float)
        node_df = node_df.drop(['Account', 'Is Laundering'], axis=1)
        node_df = df_label_encoder(node_df,['Bank'])
#         node_df = torch.from_numpy(node_df.values).to(torch.float)  # comment for visualization
        return node_df, node_label

In [13]:
node_df, node_label = get_node_attr(currency_ls, paying_df, receiving_df, accounts)
node_df

/var/folders/s7/r__8fgpd56l0s3kgg97276280000gn/T/ipykernel_86143/2334129680.py:17: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1704987089515/work/torch/csrc/utils/tensor_numpy.cpp:212.)
  node_label = torch.from_numpy(node_df['Is Laundering'].values).to(torch.float)


,Bank,avg paid 0,avg paid 1,avg paid 2,avg paid 3,avg paid 4,avg paid 5,avg paid 6,avg paid 7,avg paid 8,avg paid 9,avg paid 10,avg paid 11,avg paid 12,avg paid 13,avg paid 14,avg received 0,avg received 1,avg received 2,avg received 3,avg received 4,avg received 5,avg received 6,avg received 7,avg received 8,avg received 9,avg received 10,avg received 11,avg received 12,avg received 13,avg received 14
0,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1922.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,330.166429,0.0,0.0
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,480.223333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,119.992000,0.0,0.0
2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14675.570000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14675.570000,0.0,0.0
3,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,37340.843333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,756.486190,0.0,0.0
4,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49649.409677,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3120.573333,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515083,640,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1780.662273,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6960.194583,0.0,0.0
515084,659,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4082.224444,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3657.100000,0.0,0.0
515085,801,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,111.120000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,111.120000,0.0,0.0
515086,807,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1664.320000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,334.675000,0.0,0.0


`node_label`は各口座が不正取引口座かどうかのフラグである。

In [14]:
node_label

tensor([0., 0., 0.,  ..., 0., 0., 0.])

## エッジインデックスとエッジ特徴量

使用するカラム名: エッジインデックス`edge_index`は全口座をインデックス化して`[2, トランザクション数]`の配列とする。エッジ属性`edge_attr`には `Timestamp`, `Amount Received`, `Receiving Currency`, `Amount Paid`, `Payment Currency`, `Payment Format` のカラムを使用する。サイズは、`[トランザクション数, num of col]`である。ノード特徴量は口座単位ではあるが、エッジ情報はトランザクション単位なので注意。

In [15]:
def get_edge_df(accounts, df):
        accounts = accounts.reset_index(drop=True)
        accounts['ID'] = accounts.index
        mapping_dict = dict(zip(accounts['Account'], accounts['ID']))
        df['From'] = df['Account'].map(mapping_dict)
        df['To'] = df['Account.1'].map(mapping_dict)
        df = df.drop(['Account', 'Account.1', 'From Bank', 'To Bank'], axis=1)

        edge_index = torch.stack([torch.from_numpy(df['From'].values), torch.from_numpy(df['To'].values)], dim=0)

        df = df.drop(['Is Laundering', 'From', 'To'], axis=1)

        # edge_attr = torch.from_numpy(df.values).to(torch.float)  # comment for visualization

        edge_attr = df  # for visualization
        return edge_attr, edge_index

edge_attr, edge_index = get_edge_df(accounts, df)


`accounts = accounts.reset_index(drop=True), accounts['ID'] = accounts.index`でノードにインデックスを割り振る。

|Account | Bank | Is Laundering | ID|
|--------|-----|-----------------|---- |
|A       | B1   | 0              | 0|
|B       | B2   | 1              | 1|
|C       | B3   | 1              | 2|

`mapping_dict = dict(zip(accounts['Account'], accounts['ID']))`で`{'A': 0,'B': 1,'C': 2}`を作る。これが文字列の口座から数値IDへのマッピング表になる。

これを`df['From'] = df['Account'].map(mapping_dict), df['To'] = df['Account.1'].map(mapping_dict)`でIDに変換する。

Account | Account.1 | ... | From | To
--------|------------|-----|------|-------
A       | B         | ... | 0    | 1
B       | C         | ... | 1    | 2
A       | C         | ... | 0    | 2

不要列を削除し、`edge_index = torch.stack([torch.from_numpy(df['From'].values), torch.from_numpy(df['To'].values)], dim=0)`で  `dge_index`を作る。これでグラフの接続情報が作成される。

```Python
[
    [0,1,0]
    [1,2,2]
]
```

In [16]:
print(f'edge_index shapq: {edge_index.shape}')
print('-'*100)
edge_index

edge_index shapq: torch.Size([2, 5078345])
----------------------------------------------------------------------------------------------------


tensor([[     0,      0,      0,  ..., 496997, 496997, 496998],
        [299458, 299458, 299458,  ..., 496997, 496997, 496998]])

In [17]:
edge_attr

,Timestamp,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format
4278714,0.456320,787197.11,13,787197.11,13,3
2798190,0.285018,787197.11,13,787197.11,13,3
2798191,0.284233,681262.19,13,681262.19,13,4
3918769,0.417079,681262.19,13,681262.19,13,4
213094,0.000746,146954.27,13,146954.27,13,5
...,...,...,...,...,...,...
892752,0.040066,3481092.47,6,3481092.47,6,5
641019,0.023293,4434071.16,6,4434071.16,6,5
244392,0.000079,63015743.94,6,63015743.94,6,5
557982,0.016537,592.46,6,592.46,6,5


## モデルアーキテクチャ


ノード数$N$、ノート特徴量次元数$F$とすると、$x \in R^{N×F}$となる。モデルは

入力
 →
GATConv（多頭）
 →
GATConv（単頭）
 →
Linear
 →
Sigmoid

であり、最後に2値のノード分類を行う。

```Python
Layer                入力サイズ     出力サイズ
------------------------------------------------
Input                (N, 5)         (N, 5)
Dropout              (N, 5)         (N, 5)
GATConv1             (N, 5)         (N, 32) # Multi Heads
ELU                  (N, 32)        (N, 32)
Dropout              (N, 32)        (N, 32) # Single Head
GATConv2             (N, 32)        (N, 2)
ELU                  (N, 2)         (N, 2)
Linear               (N, 2)         (N, 1)
Sigmoid              (N, 1)         (N, 1)

# sample N=N, F=5, hidden_channels=8, heads=4

Input: (N, F) # (N, 5)
↓
self.conv1 = GATConv(in_channels, hidden_channels, heads, dropout=0.6)
↓
Output: (N, hidden × heads) # (N, 8*4) = (N, 32)
↓
Input: (N, hidden × heads) # (N, 8*4) = (N, 32)
↓
self.conv2 = GATConv(hidden_channels * heads, int(hidden_channels/4), heads=1, concat=False, dropout=0.6)
↓
Output: (N, hidden/4× heads) # (N, 8/4*1)  = (N, 2)
↓
Input: (N, hidden/4× heads) # (N, 8/4*1)  = (N, 2)
↓
self.lin = Linear(int(hidden_channels/4), out_channels)
↓
Output: (N, hidden/4) # (N, 2)
↓
Input: (N, hidden/4) # (N, 2)
↓
self.sigmoid = nn.Sigmoid()
↓
Output: (N, 1)
```

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.transforms as T
from torch_geometric.nn import GATConv, Linear

class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads, dropout=0.6)
        self.conv2 = GATConv(hidden_channels * heads, int(hidden_channels/4), heads=1, concat=False, dropout=0.6)
        self.lin = Linear(int(hidden_channels/4), out_channels)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, edge_index, edge_attr):
        x = F.dropout(x, p=0.6, training=self.training)
        x = F.elu(self.conv1(x, edge_index, edge_attr))
        x = F.dropout(x, p=0.6, training=self.training)
        x = F.elu(self.conv2(x, edge_index, edge_attr))
        x = self.lin(x)
        x = self.sigmoid(x)
        
        return x

## PyG InMemoryDataset

ここまで説明した内容をクラスにまとめておく。これ実行すればデータセットを構築できる。

In [19]:
class AMLtoGraph(InMemoryDataset):

    def __init__(self, root: str, edge_window_size: int = 10,
                 transform: Optional[Callable] = None,
                 pre_transform: Optional[Callable] = None):
        self.edge_window_size = edge_window_size
        super().__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0])

    @property
    def raw_file_names(self) -> str:
        return 'HI-Small_Trans.csv'

    @property
    def processed_file_names(self) -> str:
        return 'data.pt'

    @property
    def num_nodes(self) -> int:
        return self._data.edge_index.max().item() + 1

    def df_label_encoder(self, df, columns):
        le = preprocessing.LabelEncoder()
        for i in columns:
            df[i] = le.fit_transform(df[i].astype(str))
        return df


    def preprocess(self, df):
        df = self.df_label_encoder(df,['Payment Format', 'Payment Currency', 'Receiving Currency'])
        df['Timestamp'] = pd.to_datetime(df['Timestamp'])
        df['Timestamp'] = df['Timestamp'].apply(lambda x: x.value)
        df['Timestamp'] = (df['Timestamp']-df['Timestamp'].min())/(df['Timestamp'].max()-df['Timestamp'].min())

        df['Account'] = df['From Bank'].astype(str) + '_' + df['Account']
        df['Account.1'] = df['To Bank'].astype(str) + '_' + df['Account.1']
        df = df.sort_values(by=['Account'])
        receiving_df = df[['Account.1', 'Amount Received', 'Receiving Currency']]
        paying_df = df[['Account', 'Amount Paid', 'Payment Currency']]
        receiving_df = receiving_df.rename({'Account.1': 'Account'}, axis=1)
        currency_ls = sorted(df['Receiving Currency'].unique())

        return df, receiving_df, paying_df, currency_ls

    def get_all_account(self, df):
        ldf = df[['Account', 'From Bank']]
        rdf = df[['Account.1', 'To Bank']]
        suspicious = df[df['Is Laundering']==1]
        s1 = suspicious[['Account', 'Is Laundering']]
        s2 = suspicious[['Account.1', 'Is Laundering']]
        s2 = s2.rename({'Account.1': 'Account'}, axis=1)
        suspicious = pd.concat([s1, s2], join='outer')
        suspicious = suspicious.drop_duplicates()

        ldf = ldf.rename({'From Bank': 'Bank'}, axis=1)
        rdf = rdf.rename({'Account.1': 'Account', 'To Bank': 'Bank'}, axis=1)
        df = pd.concat([ldf, rdf], join='outer')
        df = df.drop_duplicates()

        df['Is Laundering'] = 0
        df.set_index('Account', inplace=True)
        df.update(suspicious.set_index('Account'))
        df = df.reset_index()
        return df
    
    def paid_currency_aggregate(self, currency_ls, paying_df, accounts):
        for i in currency_ls:
            temp = paying_df[paying_df['Payment Currency'] == i]
            accounts['avg paid '+str(i)] = temp['Amount Paid'].groupby(temp['Account']).transform('mean')
        return accounts

    def received_currency_aggregate(self, currency_ls, receiving_df, accounts):
        for i in currency_ls:
            temp = receiving_df[receiving_df['Receiving Currency'] == i]
            accounts['avg received '+str(i)] = temp['Amount Received'].groupby(temp['Account']).transform('mean')
        accounts = accounts.fillna(0)
        return accounts

    def get_edge_df(self, accounts, df):
        accounts = accounts.reset_index(drop=True)
        accounts['ID'] = accounts.index
        mapping_dict = dict(zip(accounts['Account'], accounts['ID']))
        df['From'] = df['Account'].map(mapping_dict)
        df['To'] = df['Account.1'].map(mapping_dict)
        df = df.drop(['Account', 'Account.1', 'From Bank', 'To Bank'], axis=1)

        edge_index = torch.stack([torch.from_numpy(df['From'].values), torch.from_numpy(df['To'].values)], dim=0)

        df = df.drop(['Is Laundering', 'From', 'To'], axis=1)

        edge_attr = torch.from_numpy(df.values).to(torch.float)
        return edge_attr, edge_index

    def get_node_attr(self, currency_ls, paying_df,receiving_df, accounts):
        node_df = self.paid_currency_aggregate(currency_ls, paying_df, accounts)
        node_df = self.received_currency_aggregate(currency_ls, receiving_df, node_df)
        node_label = torch.from_numpy(node_df['Is Laundering'].values).to(torch.float)
        node_df = node_df.drop(['Account', 'Is Laundering'], axis=1)
        node_df = self.df_label_encoder(node_df,['Bank'])
        node_df = torch.from_numpy(node_df.values).to(torch.float)
        return node_df, node_label

    def process(self):
        df = pd.read_csv(self.raw_paths[0])
        df, receiving_df, paying_df, currency_ls = self.preprocess(df)
        accounts = self.get_all_account(df)
        node_attr, node_label = self.get_node_attr(currency_ls, paying_df,receiving_df, accounts)
        edge_attr, edge_index = self.get_edge_df(accounts, df)

        data = Data(x=node_attr,
                    edge_index=edge_index,
                    y=node_label,
                    edge_attr=edge_attr
                    )
        
        data_list = [data] 
        if self.pre_filter is not None:
            data_list = [d for d in data_list if self.pre_filter(d)]

        if self.pre_transform is not None:
            data_list = [self.pre_transform(d) for d in data_list]

        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])

## モデルトレーニング


モデルの学習を行う。`NeighborLoader` は、巨大なグラフをそのまま全部読むのではなく、一部だけ切り出してミニバッチ学習する。中心ノードを選び、その周辺ノードをサンプリング、部分グラフを作る、ということをやってくれる。

In [ ]:
import torch
import torch_geometric.transforms as T
from torch_geometric.loader import NeighborLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset = AMLtoGraph('./data')
data = dataset[0]
epoch = 100

model = GAT(in_channels=data.num_features, hidden_channels=16, out_channels=1, heads=8)
model = model.to(device)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

`RandomNodeSplit`はグラフ自体は1個で全体のまま。そこからノードを割合に応じて学習、検証にわける。テストデータはここではなし。`split`することでマスクが付与されるので、後でそのマスクをもとにデータを分ける。 

`num_neighbors=[30] * 2`は、`[30, 30]`と同じで、サンプリングの範囲と密度を決める。これにより、巨大なグラフ全体を読み込む代わりに、計算に必要な「部分グラフ」だけを抽出できる。2層GNNでは2段階の近傍サンプリングが必要になる。1層目で1ホップ先の情報を使い、2層目でさらにその先の情報も必要になる。

- 1層目（自分の隣）: 各ノードから最大30個の近傍ノードをランダムに選ぶ。
- 2層目（隣の隣）: 選ばれた近傍ノードからさらに最大30個ずつ近傍ノードを選ぶ。

`batch_size=256`は、1つのバッチに含まれる中心ノードの数を256個に設定している。

In [ ]:
split = T.RandomNodeSplit(split='train_rest', num_val=0.1, num_test=0)
data = split(data)

train_loader = NeighborLoader(
    data,                        # 対象となるグラフデータ
    num_neighbors=[30] * 2,      # 各層でサンプリングする近傍ノードの数
    batch_size=256,              # 1回のリテレーションで扱う中心ノードの数
    input_nodes=data.train_mask, # サンプリングを開始するターゲットノード
)

test_loader = loader = NeighborLoader(
    data,
    num_neighbors=[30] * 2,
    batch_size=256,
    input_nodes=data.val_mask,
)

各epochでやることは次の通り。

1. `train_loader` を使って学習
2. `test_loader` を使って検証
3. `train` `loss`, `val loss`, `val accuracy` を保存
4. 検証`loss`が改善したかを見る
5. 改善しなければカウントし、一定回数で停止

`for batch in train_loader:`のデータは元の巨大グラフではなく、`NeighborLoader` が切り出した部分グラフが利用される。

In [ ]:
train_losses = []
val_losses = []
val_accs = []

patience = 10
best_val_loss = float('inf')
counter = 0

for i in range(epoch):
    total_loss = 0
    model.train()

    # ===== train =====
    # 訓練対象ノード数が1000で、バッチサイズが256なら、おおよそ4バッチ程度
    for data in train_loader:
        optimizer.zero_grad()
        data = data.to(device)

        pred = model(data.x, data.edge_index, data.edge_attr)
        ground_truth = data.y

        loss = criterion(pred, ground_truth.unsqueeze(1)) # [N,]から[N,1]に変換
        loss.backward()
        optimizer.step()

        total_loss += float(loss)

    train_loss = total_loss / len(train_loader)
    train_losses.append(train_loss)

    # ===== validation =====
    model.eval()
    val_loss = 0
    acc = 0
    total = 0

    with torch.no_grad():
        for test_data in test_loader:
            test_data = test_data.to(device)

            pred = model(test_data.x, test_data.edge_index, test_data.edge_attr)
            ground_truth = test_data.y

            loss = criterion(pred, ground_truth.unsqueeze(1))
            val_loss += float(loss)

            pred_label = (pred >= 0.5).float()
            correct = (pred_label == ground_truth.unsqueeze(1)).sum().item()

            total += len(ground_truth)
            acc += correct

    val_loss /= len(test_loader)
    acc /= total

    val_losses.append(val_loss)
    val_accs.append(acc)

    print(f"Epoch {i}, Train Loss {train_loss:.4f}, Val Loss {val_loss:.4f}, Acc {acc:.4f}")

    # Early Stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break



Epoch 0, Train Loss 3.0949, Val Loss 2.0469, Acc 0.9747
Epoch 1, Train Loss 0.8168, Val Loss 2.0222, Acc 0.9749
Epoch 2, Train Loss 0.7884, Val Loss 2.0213, Acc 0.9748
Epoch 3, Train Loss 0.7513, Val Loss 1.5337, Acc 0.9748
Epoch 4, Train Loss 0.5271, Val Loss 0.1926, Acc 0.9749
Epoch 5, Train Loss 0.2152, Val Loss 0.1039, Acc 0.9748
Epoch 6, Train Loss 0.1382, Val Loss 0.1034, Acc 0.9749
Epoch 7, Train Loss 0.1224, Val Loss 0.1038, Acc 0.9748
Epoch 8, Train Loss 0.1208, Val Loss 0.1038, Acc 0.9747
Epoch 9, Train Loss 0.1109, Val Loss 0.1039, Acc 0.9747
Epoch 10, Train Loss 0.1081, Val Loss 0.1039, Acc 0.9747
Epoch 11, Train Loss 0.1067, Val Loss 0.1037, Acc 0.9747
Epoch 12, Train Loss 0.1037, Val Loss 0.1035, Acc 0.9748
Epoch 13, Train Loss 0.1044, Val Loss 0.1038, Acc 0.9747
Epoch 14, Train Loss 0.1036, Val Loss 0.1037, Acc 0.9747
Epoch 15, Train Loss 0.1089, Val Loss 0.1040, Acc 0.9747
Epoch 16, Train Loss 0.1030, Val Loss 0.1033, Acc 0.9749
Epoch 17, Train Loss 0.1030, Val Loss 0.1

## 予測



In [50]:
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

df_new = pd.read_csv('prediction.csv')
df_new, receiving_df_new, paying_df_new, currency_ls = preprocess(df_new)
accounts_new = get_all_account(df_new)

node_df_new, _ = get_node_attr(
    currency_ls,
    paying_df_new,
    receiving_df_new,
    accounts_new
)

node_df_new = node_df_new.reindex(columns=node_df.columns, fill_value=0)
edge_attr_new, edge_index_new = get_edge_df(accounts_new, df_new)

x_new = torch.tensor(node_df_new.values, dtype=torch.float)
edge_index_new = edge_index_new.long()
edge_attr_new = torch.tensor(edge_attr_new.values, dtype=torch.float)

device = next(model.parameters()).device

model.eval()

x_new = x_new.to(device)
edge_index_new = edge_index_new.to(device)
edge_attr_new = edge_attr_new.to(device)

with torch.no_grad():
    pred = model(x_new, edge_index_new, edge_attr_new)

pred_prob = pred.cpu().numpy()
pred_label = (pred >= 0.5).float().cpu().numpy()
accounts_new["pred_prob"] = pred_prob
accounts_new["pred_label"] = pred_label

accounts_new


,Account,Bank,Is Laundering,avg paid 0,avg paid 1,avg received 0,avg received 1,pred_prob,pred_label
0,B1_B1_A,B1,0,NaN,50.0,NaN,49.0,0.019334,0.0
1,B1_B1_B,B1,0,NaN,60.0,NaN,59.0,0.019347,0.0
2,B1_B1_C,B1,0,NaN,55.0,NaN,54.0,0.019344,0.0
3,B1_B1_C1,B1,0,NaN,300.0,NaN,290.0,0.019348,0.0
4,B1_B1_C2,B1,0,NaN,290.0,NaN,280.0,0.019348,0.0
5,B1_B1_C3,B1,0,NaN,280.0,NaN,985.0,0.019348,0.0
6,B1_B1_D,B1,0,NaN,65.0,NaN,64.0,0.019348,0.0
7,B1_B1_H,B1,0,NaN,1700.0,NaN,1600.0,0.019348,0.0
8,B1_B1_M,B1,0,NaN,1700.0,NaN,1500.0,0.019348,0.0
9,B1_B1_P,B1,0,NaN,1700.0,NaN,1400.0,0.019348,0.0


In [ ]:
# ## 予測データの作成
# df_new = pd.DataFrame({
#     "Account": [
#         # 正常
#         "A","B","C","D",
#         # レイヤリング＋通貨変換
#         "X","Y","Z","W","V",
#         # スマーフィング
#         "M","M","M","M","M",
#         # 再集約
#         "P","Q","R","S","T",
#         # サイクル
#         "C1","C2","C3",
#         # ミュール
#         "U1","U2","U3",
#         # ハブ
#         "H","H","H","H","H"
#     ],

#     "Account.1": [
#         # 正常
#         "B","C","D","A",
#         # レイヤリング
#         "Y","Z","W","V","O",
#         # スマーフィング
#         "P","Q","R","S","T",
#         # 再集約
#         "Z1","Z1","Z1","Z1","Z1",
#         # サイクル
#         "C2","C3","C1",
#         # ミュール（受け取りのみ）
#         "Z2","Z3","Z4",
#         # ハブ
#         "U1","U2","U3","C1","Z1"
#     ],

#     "From Bank": ["B1"]*30,
#     "To Bank": ["B2"]*30,

#     "Amount Paid": [
#         # 正常
#         50,60,55,65,
#         # レイヤリング（減衰）
#         1000,950,920,900,880,
#         # スマーフィング（分割）
#         1000,1000,1000,1000,1000,
#         # 再集約
#         200,200,200,200,200,
#         # サイクル
#         300,290,280,
#         # ミュール
#         700,750,720,
#         # ハブ
#         1500,1600,1700,1800,1900
#     ],

#     "Amount Received": [
#         # 正常
#         49,59,54,64,
#         # レイヤリング
#         950,920,900,880,860,
#         # スマーフィング
#         200,200,200,200,200,
#         # 再集約
#         980,980,980,980,980,
#         # サイクル
#         290,280,270,
#         # ミュール（受け取り大きい）
#         680,730,700,
#         # ハブ
#         1400,1500,1600,1700,1800
#     ],

#     "Payment Currency": [
#         "USD","USD","USD","USD",
#         "USD","EUR","USD","EUR","USD",
#         "USD","USD","USD","USD","USD",
#         "EUR","EUR","EUR","EUR","EUR",
#         "USD","USD","USD",
#         "USD","USD","USD",
#         "USD","USD","USD","USD","USD"
#     ],

#     "Receiving Currency": [
#         "USD","USD","USD","USD",
#         "EUR","USD","EUR","USD","EUR",
#         "USD","USD","USD","USD","USD",
#         "EUR","EUR","EUR","EUR","EUR",
#         "USD","USD","USD",
#         "USD","USD","USD",
#         "USD","USD","USD","USD","USD"
#     ],

#     "Payment Format": ["Wire"]*30,

#     "Timestamp": [
#         # 正常
#         "2024-01-01","2024-01-01","2024-01-02","2024-01-02",
#         # レイヤリング（連続）
#         "2024-01-03","2024-01-03","2024-01-03","2024-01-04","2024-01-04",
#         # スマーフィング（同日）
#         "2024-01-05","2024-01-05","2024-01-05","2024-01-05","2024-01-05",
#         # 再集約
#         "2024-01-06","2024-01-06","2024-01-06","2024-01-06","2024-01-06",
#         # サイクル
#         "2024-01-07","2024-01-07","2024-01-07",
#         # ミュール
#         "2024-01-08","2024-01-08","2024-01-08",
#         # ハブ
#         "2024-01-09","2024-01-09","2024-01-09","2024-01-09","2024-01-09"
#     ],

#     "Is Laundering": [0]*30
# })


In [ ]:
# import matplotlib.pyplot as plt

# # Loss
# plt.figure()
# plt.plot(train_losses, label="Train Loss")
# plt.plot(val_losses, label="Val Loss")
# plt.xlabel("Epoch")
# plt.ylabel("Loss")
# plt.legend()
# plt.title("Loss Curve")
# plt.show()

# # Accuracy
# plt.figure()
# plt.plot(val_accs, label="Val Accuracy")
# plt.xlabel("Epoch")
# plt.ylabel("Accuracy")
# plt.legend()
# plt.title("Accuracy Curve")
# plt.show()